<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.4：时序逻辑
**上一步：[控制流](2.3_control_flow.ipynb)**<br>
**下一步：[FIR 滤波器](2.5_exercise.ipynb)**

## 动机
没有状态，你就无法编写任何有意义的数字逻辑。没有状态，你就无法编写任何有意义的数字逻辑。没有状态，你就无法编写任何有意义的数字逻辑……

明白了吗？因为不存储中间结果，你就无法取得任何进展。

好了，不开玩笑了，本模块将描述如何在 Chisel 中表达常见的时序模式。在本模块结束时，您应该能够在 Chisel 中实现并测试一个移位寄存器。

需要强调的是，本节可能不会给您留下深刻印象。Chisel 的强大之处不在于新的时序逻辑模式，而在于设计的参数化。在演示该功能之前，我们必须先了解这些时序模式是什么。因此，本节将向您展示 Chisel 几乎可以完成 Verilog 所能完成的所有事情——您只需要学习 Chisel 语法即可。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 寄存器
Chisel 中的基本状态元件是寄存器，表示为 `Reg`。
`Reg` 会保持其输出值，直到其时钟的上升沿，此时它会获取其输入的值。
默认情况下，每个 Chisel `Module` 都有一个隐式时钟，设计中的每个寄存器都使用该时钟。
这样可以避免您总是在代码中指定相同的时钟。

<span style="color:blue">**示例：使用寄存器**</span><br>
以下代码块实现了一个模块，该模块获取输入，将其加 1，然后将其连接为寄存器的输入。
*注意：对于多时钟设计，可以覆盖隐式时钟。有关示例，请参见附录。*

In [ ]:
class RegisterModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  val register = Reg(UInt(12.W))
  register := io.in + 1.U
  io.out := register
}

test(new RegisterModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("成功！！")

寄存器通过调用 `Reg(tpe)` 创建，其中 `tpe` 是一个编码我们想要的寄存器类型的变量。
在此示例中，`tpe` 是一个 12 位 `UInt`。

看看上面的测试程序在做什么。
在调用 `poke()` 和 `expect` 之间，有一个对 `step(1)` 的调用。
这告诉测试工具将时钟跳动一次，这将导致寄存器将其输入传递到其输出。

调用 `step(n)` 将使时钟跳动 `n` 次。

细心的观察者会注意到，以前测试组合逻辑的测试程序没有调用 `step()`。这是因为在输入上调用 `poke()` 会立即通过组合逻辑传播更新的值。只有在更新时序逻辑中的状态元素时才需要调用 `step()`。

下面的代码块将显示由 `RegisterModule` 生成的 verilog。

注意：
* 该模块有一个您未添加的时钟（和复位）输入——这是隐式时钟
* 变量 `register` 按预期显示为 `reg [11:0]`
* 有一个由 `ifdef Randomize` 分隔的块，在仿真开始前将寄存器初始化为某个随机变量
* `register` 在 `posedge clock` 更新

In [ ]:
println(getVerilog(new RegisterModule))

一个重要的注意事项是，Chisel 区分类型（如 `UInt`）和硬件节点（如文字 `2.U` 或 `myReg` 的输出）。虽然
```scala
val myReg = Reg(UInt(2.W))
```
是合法的，因为 Reg 需要一个数据类型作为模型，
```scala
val myReg = Reg(2.U)
```
是一个错误，因为 `2.U` 已经是一个硬件节点，不能用作模型。

<span style="color:blue">**示例：RegNext**</span><br>
Chisel 有一个方便的寄存器对象，用于具有简单输入连接的寄存器。前面的 `Module` 可以缩短为下面的 `Module`。请注意，这次我们不需要指定寄存器位宽。它是从寄存器的输出连接（在本例中为 `io.out`）推断出来的。

In [ ]:
class RegNextModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  // 寄存器位宽由 io.out 推断得出
  io.out := RegNext(io.in + 1.U)
}

test(new RegNextModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("成功！！")

Verilog 代码与之前几乎相同，只是寄存器名称是生成的，而不是显式定义的。

In [ ]:
println(getVerilog(new RegNextModule))

---
# `RegInit`

`RegisterModule` 中的寄存器在仿真时被初始化为随机数据。
除非另有说明，否则寄存器没有复位值（或复位）。
创建复位到给定值的寄存器的方法是使用 `RegInit`。

例如，可以使用以下代码创建一个初始化为零的 12 位寄存器。
以下两个版本都有效并且执行相同的操作：
```scala
val myReg = RegInit(UInt(12.W), 0.U)
val myReg = RegInit(0.U(12.W))
```

第一个版本有两个参数。
第一个参数是一个类型节点，指定数据类型及其宽度。
第二个参数是一个硬件节点，指定复位值，在本例中为 0。

第二个版本有一个参数。
它是一个硬件节点，指定复位值，但通常是 `0.U`。

<span style="color:blue">**示例：初始化寄存器** </span><br>
下面演示了使用 `RegInit()`，初始化为零。

In [ ]:
class RegInitModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  val register = RegInit(0.U(12.W))
  register := io.in + 1.U
  io.out := register
}

println(getVerilog(new RegInitModule))

请注意，生成的 verilog 现在有一个检查 `if (reset)` 的块，以将寄存器复位为 0。
另请注意，这位于 `always @(posedge clock)` 块内。
Chisel 的隐式复位是高电平有效且同步的。
在调用复位之前，寄存器仍会初始化为随机垃圾值。
`PeekPokeTesters` 在运行测试之前总是调用复位，但您也可以使用 `reset(n)` 函数手动调用复位，其中复位在高电平持续 `n` 个周期。

---
# 控制流
寄存器在控制流方面与线非常相似。
它们具有最后连接语义，并且可以使用 `when`、`elsewhen` 和 `otherwise` 进行条件赋值。

<span style="color:blue">**示例：寄存器控制流**</span><br>
以下示例使用条件寄存器赋值在一系列输入中查找最大值。

In [ ]:
class FindMax extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(10.W))
    val max = Output(UInt(10.W))
  })

  val max = RegInit(0.U(10.W))
  when (io.in > max) {
    max := io.in
  }
  io.max := max
}

test(new FindMax) { c =>
    c.io.max.expect(0.U)
    c.io.in.poke(1.U)
    c.clock.step(1)
    c.io.max.expect(1.U)
    c.io.in.poke(3.U)
    c.clock.step(1)
    c.io.max.expect(3.U)
    c.io.in.poke(2.U)
    c.clock.step(1)
    c.io.max.expect(3.U)
    c.io.in.poke(24.U)
    c.clock.step(1)
    c.io.max.expect(24.U)
}
println("成功！！")

---
# 其他寄存器示例

在寄存器上调用的操作是在寄存器的**输出**上执行的，操作的类型取决于寄存器的类型。
这意味着您可以编写
```scala
val reg: UInt = Reg(UInt(4.W))
```
这意味着值 `reg` 的类型为 `UInt`，您可以执行通常可以对 `UInt` 执行的操作，例如 `+`、`-` 等。


您不仅限于将 `UInt` 与寄存器一起使用，还可以使用基本类型 `chisel3.Data` 的任何子类。这包括用于有符号整数的 `SInt` 和许多其他东西。

<span style="color:blue">**示例：梳状滤波器**</span><br>
以下示例显示了一个梳状滤波器。

In [ ]:
class Comb extends Module {
  val io = IO(new Bundle {
    val in  = Input(SInt(12.W))
    val out = Output(SInt(12.W))
  })

  val delay: SInt = Reg(SInt(12.W))
  delay := io.in
  io.out := io.in - delay
}
println(getVerilog(new Comb))

---
# 练习
<span style="color:red">**练习：移位寄存器**</span><br>
利用您新掌握的寄存器知识，构建一个实现 LFSR 移位寄存器的模块。具体来说：
- 每个元素都是单位宽度的。
- 有一个 4 位输出信号。
- 接受一个单位输入位，这是移位寄存器的下一个值。
- 输出移位寄存器的并行输出，最高有效位是移位寄存器的最后一个元素，最低有效位是移位寄存器的第一个元素。`Cat` 可能会派上用场。
- **输出初始化为 `b0001`。**
- 每个时钟周期移位（无使能信号）。
- **注意，在 Chisel 中，子字赋值是非法的**；类似 `out(0) := in` 的代码将不起作用。

<img src="images/shifter4.svg" alt="移位寄存器图" style="width: 450px" />

下面提供了一个基本的模块骨架、测试向量和驱动程序调用。第一个寄存器已为您提供。

In [ ]:
class MyShiftRegister(val init: Int = 1) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Bool())
    val out = Output(UInt(4.W))
  })

  val state = RegInit(UInt(4.W), init.U)

  ???
}

test(new MyShiftRegister()) { c =>
  var state = c.init
  for (i <- 0 until 10) {
    // 输入 i 的最低有效位 (i % 2)
    c.io.in.poke(((i % 2) != 0).B)
    // 更新预期状态
    state = ((state * 2) + (i % 2)) & 0xf
    c.clock.step(1)
    c.io.out.expect(state.U)
  }
}
println("成功！！")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  state := nextState
  io.out := state
</pre></article></div></section></div>

<span style="color:red">**练习：参数化移位寄存器（可选）**</span><br>
编写一个移位寄存器，该寄存器由其延迟 (`n`)、其初始值 (`init`) 参数化，并且还具有一个使能输入信号 (`en`)。

In [ ]:
// n 是输出宽度（延迟数 - 1）
// 将初始状态设置为 init
class MyOptionalShiftRegister(val n: Int, val init: BigInt = 1) extends Module {
  val io = IO(new Bundle {
    val en  = Input(Bool())
    val in  = Input(Bool())
    val out = Output(UInt(n.W))
  })

  val state = RegInit(init.U(n.W))

  ???
}

// 测试不同的深度
for (i <- Seq(3, 4, 8, 24, 65)) {
  println(s"测试 n=$i")
  test(new MyOptionalShiftRegister(n = i)) { c =>
    val inSeq = Seq(0, 1, 1, 1, 0, 1, 1, 0, 0, 1)
    var state = c.init
    var i = 0
    c.io.en.poke(true.B)
    while (i < 10 * c.n) {
      // 输入重复的 inSeq
      val toPoke = inSeq(i % inSeq.length)
      c.io.in.poke((toPoke != 0).B)
      // 更新预期状态
      state = ((state * 2) + toPoke) & BigInt("1"*c.n, 2)
      c.clock.step(1)
      c.io.out.expect(state.U)

      i += 1
    }
  }
}
println("成功！！")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  when (io.en) {
    state  := nextState
  }
  io.out := state
</pre></article></div></section></div>

---
# 附录：显式时钟和复位
Chisel 模块具有默认时钟和复位，模块内部创建的每个寄存器都隐式使用它们。
有时您希望能够覆盖此默认行为；例如，您可能有一个生成时钟或复位信号的黑盒，或者您有一个多时钟设计。

Chisel 提供了处理这些情况的结构。
可以使用 `withClock() {}`、`withReset() {}` 和 `withClockAndReset() {}` 单独或一起覆盖时钟和复位。
以下代码块将给出使用这些函数的示例。

需要注意的一点是，`reset`（在本教程编写时）始终是同步的，类型为 `Bool`。
时钟在 Chisel 中有自己的类型 (`Clock`)，应如此声明。
*`Bool` 可以通过在其上调用 `asClock()` 转换为 `Clock`，但您应该小心不要做一些愚蠢的事情。*

另请注意，`chisel-testers` 目前不完全支持多时钟设计。

<span style="color:blue">**示例：多时钟模块**</span><br>
具有多个时钟和复位信号的模块。

In [ ]:
// 我们需要导入多时钟功能
import chisel3.experimental.{withClock, withReset, withClockAndReset}

class ClockExamples extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(10.W))
    val alternateReset    = Input(Bool())
    val alternateClock    = Input(Clock())
    val outImplicit       = Output(UInt())
    val outAlternateReset = Output(UInt())
    val outAlternateClock = Output(UInt())
    val outAlternateBoth  = Output(UInt())
  })

  val imp = RegInit(0.U(10.W))
  imp := io.in
  io.outImplicit := imp

  withReset(io.alternateReset) {
    // 此范围内的所有内容都将 alternateReset 作为复位
    val altRst = RegInit(0.U(10.W))
    altRst := io.in
    io.outAlternateReset := altRst
  }

  withClock(io.alternateClock) {
    val altClk = RegInit(0.U(10.W))
    altClk := io.in
    io.outAlternateClock := altClk
  }

  withClockAndReset(io.alternateClock, io.alternateReset) {
    val alt = RegInit(0.U(10.W))
    alt := io.in
    io.outAlternateBoth := alt
  }
}

println(getVerilog(new ClockExamples))

---
# 总结
恭喜您完成本节！！您现在已经学会了如何在 Chisel 中创建寄存器和编写时序逻辑，这意味着您拥有足够的构建模块来编写实际电路。

下一节将把我们学到的所有内容整合到一个示例中！如果您需要更多鼓励，请记住一位 Chisel 专家用户的话：

![BobRoss](http://i.qkme.me/3qbd5u.jpg)

---
# 您已完成！

[返回顶部。](#top)